# Chapter 5: Foundational Cognitive Architectures

**Book:** *30 Agents Every AI Engineer Must Build*  
**Author:** Imran Ahmad  
**Publisher:** Packt Publishing, 2026  
**Chapter Pages:** 117–144

> *"Information is the resolution of uncertainty."*  
> — **Claude Shannon**, Father of Information Theory

---

## Introduction

This chapter examines the three foundational cognitive architectures that empower intelligent, autonomous agents to operate in complex, dynamic environments. These architectures serve as the structural backbone of modern AI systems, simulating essential aspects of human cognition — **decision-making**, **planning**, and **memory** — to enable agents to tackle real-world problems with increasing sophistication.

Each architecture plays a distinct role in shaping intelligent behavior. Together, they enable AI systems to evolve from simple, reactive scripts into adaptive, autonomous workflows capable of handling intricate tasks over extended time horizons. Understanding these foundational patterns is crucial because they form the backbone of virtually every intelligent system in production today.

### The Three Foundational Architectures

| Agent Type | Focus | Chapter Section | Book Pages |
|---|---|---|---|
| **Autonomous Decision-Making Agent** | Real-time perception → cognition → action loop | Section 5.1 | pp. 118–132 |
| **Planning Agent** | Hierarchical task decomposition and dynamic execution | Section 5.2 | pp. 131–137 |
| **Memory-Augmented Agent** | Working, episodic, and semantic memory for continuity | Section 5.3 | pp. 135–144 |

### Key Concepts at a Glance

| Concept | Description |
|---------|-------------|
| **Cognitive Loop** | Perception → Cognition → Action → Learning — the continuous cycle enabling autonomy (Figure 5.1) |
| **Strategy Scoring** | Weighted multi-axis framework selecting between full autonomous, immediate escalation, and guided resolution |
| **Task DAGs** | Dependency-aware directed acyclic graphs for billing, outage, and generic workflows |
| **Hierarchical Decomposition** | Breaking high-level goals into phased, actionable subtasks (Figure 5.2) |
| **Memory Triad** | Working memory (session), episodic memory (history), semantic memory (facts) — see Figure 5.3 |
| **Integrated Architecture** | All three agents combined in a single scenario (Figure 5.4, Table 5.2) |


## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [Autonomous Decision-Making Agent](#2.-Autonomous-Decision-Making-Agent) (Section 5.1, pp. 118-132)
   - Perception Module
   - Cognition Module (Strategy Scoring)
   - Action Module (Task DAG Execution)
   - Safety Checks & Escalation
   - Full Agent Class & Case Study
3. [Planning Agent](#3.-Planning-Agent) (Section 5.2, pp. 131-137)
   - Hierarchical Decomposition
   - Dynamic Execution & Monitoring
   - Marketing Campaign Demo
4. [Memory-Augmented Agent](#4.-Memory-Augmented-Agent) (Section 5.3, pp. 135-144)
   - Three Memory Types
   - Healthcare Assistant Case Study
5. [Comparative Analysis](#5.-Comparative-Analysis) (Section 5.4, pp. 141-142)
6. [Engineering Best Practices & Wrap-Up](#6.-Engineering-Best-Practices-&-Wrap-Up) (Section 5.5, pp. 142-144)

---
## 1. Environment Setup

Import standard library modules and the three supporting files that
power this notebook: `color_logger.py` (observability), `resilience.py`
(fault tolerance), and `mock_llm.py` (simulation mode).

Ref: Technical Requirements (p. 118)

In [ ]:
# Standard library imports
# Ref: Technical Requirements, p. 118
import os
import sys
import time
from datetime import datetime
from dataclasses import dataclass, field
from functools import wraps

# Ensure local modules are importable regardless of working directory
sys.path.insert(0, ".")

# Supporting modules (must exist alongside this notebook)
from color_logger import log_info, log_success, log_error, log_warn, log_section
from resilience import fail_gracefully
from mock_llm import MockLLM, MockVectorDB, MockResponse

log_success("All imports successful")

In [ ]:
# Multi-provider LLM support (OpenAI / Anthropic / Google Gemini)
# Set LLM_PROVIDER in .env to choose: openai | anthropic | google | auto
# Auto-detection uses the first available key.
# See supporting/llm_provider.py for details.

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')

try:
    from supporting.llm_provider import detect_provider, get_llm, PROVIDER_MODELS, print_provider_banner
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = detect_provider()
    print_provider_banner(_PROVIDER, _PROVIDER_MODE)
except ImportError:
    print('[INFO] supporting/llm_provider.py not found — using default OpenAI path')
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = 'openai', os.getenv('OPENAI_API_KEY'), 'LIVE' if os.getenv('OPENAI_API_KEY') else 'SIMULATION'


In [ ]:
# ── Environment Detection & Simulation Mode Toggle ─────────────
# Ref: Technical Requirements, p. 118
#
# If an API key is found in .env or via getpass, live mode activates.
# Otherwise, Simulation Mode engages the MockLLM and MockVectorDB.

log_section("Environment Detection", chapter_ref="Technical Requirements, p. 118")

# Attempt to load .env (no-op if file missing)
try:
    from dotenv import load_dotenv
    load_dotenv()
    log_info("Loaded .env file")
except ImportError:
    log_warn("python-dotenv not installed — skipping .env loading")

# Check for API key
api_key = os.getenv("OPENAI_API_KEY", "").strip()

if not api_key:
    try:
        import getpass
        # In non-interactive environments, this will remain empty
        api_key = getpass.getpass(
            "Enter OpenAI API key (or press Enter for Simulation Mode): "
        ).strip()
    except (EOFError, OSError, Exception):
        api_key = ""

# ── Initialize clients based on mode ───────────────────────────
SIMULATION_MODE = not bool(api_key)

if SIMULATION_MODE:
    llm_client = MockLLM()
    vector_db = MockVectorDB()
    log_warn("SIMULATION MODE active — using MockLLM and MockVectorDB")
    log_info("No API key required. All agent demos will run with mock data.")
else:
    # Live mode: detect provider and show banner
    try:
        sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..'))
        from supporting.llm_provider import detect_provider, get_client, chat_completion, PROVIDER_MODELS, print_provider_banner
        _provider = os.environ.get("LLM_PROVIDER", "auto").strip().lower()
        if _provider == "auto":
            _provider, _, _ = detect_provider()
        _model = PROVIDER_MODELS.get(_provider, {}).get("default", "gpt-4o")
        _real_client = get_client(provider=_provider, api_key=api_key)
        print_provider_banner(_provider, "LIVE", _model)

        class LiveLLM:
            """Wraps real LLM to match MockLLM.chat() interface."""
            def __init__(self, client, provider, model):
                self._client, self._provider, self._model = client, provider, model
            def chat(self, messages, **kwargs):
                text = chat_completion(self._client, self._provider, messages, model=self._model)
                return MockResponse(content=text, model=self._model)
            def __repr__(self):
                return f"LiveLLM({self._provider}, {self._model})"

        llm_client = LiveLLM(_real_client, _provider, _model)
        log_success(f"Live LLM: {llm_client}")
    except Exception as e:
        llm_client = MockLLM()
        log_warn(f"Live LLM init failed ({e}). Using MockLLM.")
    vector_db = MockVectorDB()

### How Simulation Mode Works

The mock layer (`mock_llm.py`) provides two drop-in replacements:

- **`MockLLM`** — Routes prompts through a keyword map to one of 6
  pre-built `MockResponse` objects covering service outages, billing
  issues, complex escalations, marketing campaigns, healthcare queries,
  and a default fallback.

- **`MockVectorDB`** — An in-memory store with Jaccard word-overlap
  scoring, pre-seeded with 4 episodic records (3 healthcare +
  1 customer service). Supports `add()`, `search()`, and `reset()`.

This design means **every cell in this notebook runs without any API key**.
To switch to live mode, simply add your key to `.env` and restart the kernel.

Ref: Technical Requirements (p. 118)

---
## 2. Autonomous Decision-Making Agent

**Chapter Ref:** Section 5.1 (pp. 118–132)

The most fundamental architecture in the cognitive hierarchy. These agents perceive their environment, evaluate possible actions against goals and constraints, choose what to do, and execute — **without requiring continuous human supervision**. They learn from outcomes by updating state or memories that inform the next loop.

The architecture implements four integrated phases: **Perception** (structuring raw inputs), **Cognition** (strategic reasoning and strategy selection), **Action** (task DAG execution), and **Learning** (feedback-driven adaptation).

### Figure 5.1 — The Autonomous Agent Decision Loop


In [ ]:
# ── Figure 5.1 — The Autonomous Agent Decision Loop (SVG) ──
# Ref: Section 5.1, p. 125

from IPython.display import SVG, display

fig_5_1 = """
<svg viewBox="0 0 500 350" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif">
  <defs><marker id="a51" markerWidth="10" markerHeight="7" refX="10" refY="3.5" orient="auto"><polygon points="0 0,10 3.5,0 7" fill="#555"/></marker></defs>
  <rect width="500" height="350" rx="12" fill="#f8f9fa" stroke="#dee2e6"/>
  <!-- Inputs label -->
  <text x="250" y="30" text-anchor="middle" font-size="12" fill="#1565c0" font-weight="bold">Inputs</text>
  <line x1="250" y1="35" x2="250" y2="55" stroke="#1565c0" stroke-width="2" marker-end="url(#a51)"/>
  <!-- Perception -->
  <rect x="100" y="60" width="300" height="55" rx="8" fill="#e3f2fd" stroke="#42a5f5" stroke-width="2.5"/>
  <text x="250" y="93" text-anchor="middle" font-size="20" font-weight="bold" fill="#1565c0">Perception</text>
  <!-- Arrow P→C -->
  <line x1="250" y1="115" x2="250" y2="135" stroke="#555" stroke-width="2" marker-end="url(#a51)"/>
  <!-- Cognition -->
  <rect x="100" y="140" width="300" height="55" rx="8" fill="#fff3e0" stroke="#ffa726" stroke-width="2.5"/>
  <text x="250" y="173" text-anchor="middle" font-size="20" font-weight="bold" fill="#e65100">Cognition</text>
  <!-- Arrow C→A -->
  <line x1="250" y1="195" x2="250" y2="215" stroke="#555" stroke-width="2" marker-end="url(#a51)"/>
  <!-- Action -->
  <rect x="100" y="220" width="300" height="55" rx="8" fill="#e8f5e9" stroke="#66bb6a" stroke-width="2.5"/>
  <text x="250" y="253" text-anchor="middle" font-size="20" font-weight="bold" fill="#2e7d32">Action</text>
  <!-- Outputs -->
  <text x="250" y="300" text-anchor="middle" font-size="12" fill="#2e7d32" font-weight="bold">Outputs</text>
  <!-- Feedback loop -->
  <path d="M 250 310 Q 250 330 420 330 Q 460 330 460 170 Q 460 45 310 45 L 270 45" fill="none" stroke="#999" stroke-width="1.5" stroke-dasharray="6,3" marker-end="url(#a51)"/>
  <!-- Feedback arrow back from Action to Perception -->
  <path d="M 100 250 Q 50 250 50 165 Q 50 80 100 80" fill="none" stroke="#42a5f5" stroke-width="2" marker-end="url(#a51)"/>
  <text x="35" y="168" text-anchor="middle" font-size="9" fill="#42a5f5" transform="rotate(-90, 35, 168)">Learning Loop</text>
  <!-- Caption -->
  <text x="250" y="345" text-anchor="middle" font-size="10" fill="#777" font-style="italic">Figure 5.1 — The autonomous agent decision loop (p. 125)</text>
</svg>
"""
display(SVG(fig_5_1))


In [ ]:
# ── Perception Module ──────────────────────────────────────────
# Ref: Section 5.1, Perception: From raw input to structured
#      intelligence (pp. 119-121)
#
# The enhanced perception system goes beyond simple input processing;
# it provides the agent with situational awareness that enables
# autonomous decision-making. When a user reports an issue, the
# agent considers system load, time of day, user priority, and
# current alerts to make informed decisions.

log_section("Perception Module", chapter_ref="Section 5.1, pp. 119-121")


@fail_gracefully(
    fallback_value={"message": "", "error": "perception_failed"},
    chapter_ref="Section 5.1, pp. 119-121",
)
def analyze_sentiment(text: str) -> str:
    """Return sentiment label for the given text.

    Args:
        text: User message to analyze.

    Returns:
        Sentiment label: 'positive', 'neutral', or 'negative'.

    Chapter Reference:
        Section 5.1, Perception (p. 119)
    """
    # Ref: p. 119 — classify_text_sentiment stub
    negative_words = {"down", "slow", "broken", "issue", "problem",
                      "can't", "error", "intermittent", "frustrated"}
    positive_words = {"thanks", "great", "helped", "resolved", "good"}
    words = set(text.lower().split())
    if words & negative_words:
        return "negative"
    if words & positive_words:
        return "positive"
    return "neutral"


@fail_gracefully(
    fallback_value=[],
    chapter_ref="Section 5.1, p. 120",
)
def check_system_alerts() -> list:
    """Return list of active system alert IDs.

    Chapter Reference:
        Section 5.1, Perception — check_system_alerts() (p. 120)
    """
    # Simulated system alerts
    alerts = ["NET-2026-0042: Regional connectivity degradation"]
    log_info(f"System alerts retrieved: {len(alerts)} active")
    return alerts


@fail_gracefully(
    fallback_value="standard",
    chapter_ref="Section 5.1, p. 120",
)
def get_user_tier(user_id: str) -> str:
    """Return account tier for the given user.

    Args:
        user_id: Customer identifier.

    Returns:
        Tier string: 'standard', 'premium', or 'enterprise'.

    Chapter Reference:
        Section 5.1, Perception — get_user_tier() (p. 120)
    """
    # Simulated CRM lookup
    tiers = {
        "premium_biz_123": "premium",
        "enterprise_500": "enterprise",
        "patient_42": "standard",
    }
    tier = tiers.get(user_id, "standard")
    log_info(f"User tier for '{user_id}': {tier}")
    return tier


@fail_gracefully(
    fallback_value=True,
    chapter_ref="Section 5.1, p. 120",
)
def check_human_agent_status() -> bool:
    """Check whether human agents are available for escalation.

    Returns:
        True if at least one human agent is online.

    Chapter Reference:
        Section 5.1, Perception — agent_availability (p. 120)
    """
    log_info("Human agent availability: True (simulated)")
    return True


@fail_gracefully(
    fallback_value={"message": "", "error": "perception_failed"},
    chapter_ref="Section 5.1, pp. 119-121",
)
def enhanced_perception(
    user_message: str,
    context: dict,
    system_state: dict,
) -> dict:
    """Build a rich perception dict from user message + environment.

    Combines the base perception (message, timestamp, sentiment)
    with system-wide context (load, alerts, user tier, agent
    availability) to enable autonomous decision-making.

    Args:
        user_message: Raw user input text.
        context: Session context dict with 'user_id', 'session' keys.
        system_state: Dict with 'active_sessions' and other runtime data.

    Returns:
        Structured perception dict with 10+ fields.

    Chapter Reference:
        Section 5.1, enhanced_perception() (pp. 120-121)
    """
    user_id = context.get("user_id", "unknown")

    # Base perception (Ref: pp. 135-136)
    perception = {
        "message": user_message,
        "timestamp": datetime.now(),
        "user_id": user_id,
        "session_state": context.get("session", "new"),
        "sentiment": analyze_sentiment(user_message),
    }

    # Enhanced with environmental awareness (Ref: p. 120)
    perception.update({
        "current_load": system_state.get("active_sessions", 0),
        "time_of_day": datetime.now().hour,
        "user_tier": get_user_tier(user_id),
        "recent_issues": check_system_alerts(),
        "agent_availability": check_human_agent_status(),
    })

    log_success(
        f"Perception complete: intent signal='{perception['sentiment']}', "
        f"tier='{perception['user_tier']}', "
        f"alerts={len(perception['recent_issues'])}"
    )
    return perception


# ── Quick test ─────────────────────────────────────────────────
test_perception = enhanced_perception(
    user_message="My business internet has been intermittent for two days",
    context={"user_id": "premium_biz_123", "session": "new"},
    system_state={"active_sessions": 42},
)
log_info(f"Perception keys: {list(test_perception.keys())}")

> **📝 Note — Strategy Scoring Framework (pp. 121–122)**  
> The agent scores each candidate strategy across four weighted axes: **autonomy_level_score** (0–1), **urgency_score** (0–1), **complexity_score** (0–1), and **escalation_threshold** (0–1). The strategy with the highest composite score wins; ties favour guided resolution. The three strategies are: *full autonomous resolution*, *immediate escalation*, and *guided autonomous resolution* (baseline fallback at 0.5).

> **📝 Note — Task DAGs (pp. 123–124)**  
> Action plans are built as dependency-aware task DAGs (directed acyclic graphs). Each node has an `id`, `action`, and `depends_on` list. Tasks with empty `depends_on` run immediately; downstream tasks wait until all listed IDs complete. The chapter provides three DAG templates: billing issues (8 tasks), service outages (6 tasks), and a generic fallback (4 tasks).


In [ ]:
# ── Cognition Module: Strategic Reasoning ──────────────────────
# Ref: Section 5.1, Cognition: Strategic reasoning for autonomous
#      operation (pp. 121-122)
#
# The reasoning component evaluates three candidate strategies
# across weighted axes (autonomy, urgency, complexity, escalation).
# The strategy with the highest composite score wins; ties favour
# guided resolution.

log_section("Cognition Module", chapter_ref="Section 5.1, pp. 121-122")


@fail_gracefully(
    fallback_value="general_inquiry",
    chapter_ref="Section 5.1, p. 121",
)
def classify_intent(message: str) -> str:
    """Classify user message intent using LLM or keyword heuristic.

    Args:
        message: User message text.

    Returns:
        Intent label string.

    Chapter Reference:
        Section 5.1, Cognition — classify_intent() (p. 121)
    """
    response = llm_client.generate(message)
    log_info(f"Intent classified: '{response.intent}' "
             f"(confidence: {response.confidence})")
    return response.intent


@fail_gracefully(
    fallback_value=3,
    chapter_ref="Section 5.1, p. 121",
)
def determine_priority(
    intent: str,
    sentiment: str,
    user_history: dict | None = None,
) -> int:
    """Determine issue priority on a 1-5 scale.

    Args:
        intent: Classified intent label.
        sentiment: Sentiment from perception ('positive'/'neutral'/'negative').
        user_history: Optional user history dict.

    Returns:
        Priority integer (1=lowest, 5=highest).

    Chapter Reference:
        Section 5.1, Cognition — determine_priority() (p. 121)
    """
    base = {"service_outage": 4, "billing_issue": 3, "complex_technical": 5}
    priority = base.get(intent, 2)
    if sentiment == "negative":
        priority = min(priority + 1, 5)
    log_info(f"Priority determined: {priority}/5 "
             f"(intent='{intent}', sentiment='{sentiment}')")
    return priority


@fail_gracefully(
    fallback_value="guided_autonomous_resolution",
    chapter_ref="Section 5.1, p. 121",
)
def determine_autonomy_level(
    user_tier: str,
    current_load: int,
    agent_availability: bool,
) -> float:
    """Compute autonomy level score [0.0-1.0].

    Higher tier + higher load + lower agent availability
    = higher autonomy level.

    Chapter Reference:
        Section 5.1, Cognition — autonomy_level (p. 121)
    """
    tier_scores = {"enterprise": 0.9, "premium": 0.8, "standard": 0.5}
    tier_score = tier_scores.get(user_tier, 0.5)
    load_factor = min(current_load / 100, 1.0)
    availability_factor = 0.3 if not agent_availability else 0.0
    return round(min(tier_score + load_factor * 0.2 + availability_factor, 1.0), 2)


@fail_gracefully(
    fallback_value=0.5,
    chapter_ref="Section 5.1, p. 121",
)
def calculate_escalation_threshold(
    time_of_day: int,
    intent: str,
    priority: int,
) -> float:
    """Calculate the escalation threshold [0.0-1.0].

    Lower threshold = easier to escalate.

    Chapter Reference:
        Section 5.1, Cognition — escalation_threshold (p. 121)
    """
    base = 0.5
    if priority >= 4:
        base -= 0.15
    if time_of_day < 8 or time_of_day > 20:  # After hours
        base -= 0.1
    return round(max(base, 0.1), 2)


@fail_gracefully(
    fallback_value={
        "intent": "general_inquiry",
        "strategy": "guided_autonomous_resolution",
    },
    chapter_ref="Section 5.1, pp. 121-122",
)
def autonomous_reasoning(perception_data: dict) -> dict:
    """Autonomous cognition: classify, prioritize, select strategy.

    Implements the 3-strategy scoring system from the book:
    - full_autonomous_resolution
    - immediate_escalation
    - guided_autonomous_resolution (baseline fallback at 0.5)

    The strategy with the highest composite score wins.

    Args:
        perception_data: Structured dict from enhanced_perception().

    Returns:
        Decision context dict with intent, priority, strategy,
        scores, and all supporting data.

    Chapter Reference:
        Section 5.1, autonomous_reasoning() with strategy scoring
        (pp. 121-122)
    """
    # Basic intent and priority (Ref: p. 121)
    intent = classify_intent(perception_data["message"])
    priority = determine_priority(
        intent,
        perception_data.get("sentiment", "neutral"),
    )

    # Autonomy and escalation parameters (Ref: p. 121)
    autonomy_level = determine_autonomy_level(
        perception_data.get("user_tier", "standard"),
        perception_data.get("current_load", 0),
        perception_data.get("agent_availability", True),
    )
    escalation_threshold = calculate_escalation_threshold(
        perception_data.get("time_of_day", 12),
        intent,
        priority,
    )

    # Build decision context
    decision_context = {
        "intent": intent,
        "priority": priority,
        "context": perception_data,
        "autonomy_level_score": autonomy_level,
        "escalation_threshold": escalation_threshold,
        "urgency_score": priority / 5.0,
        "complexity_score": 0.85 if intent == "complex_technical" else 0.3,
        "agent_availability": perception_data.get("agent_availability", True),
    }

    # ── Strategy scoring (Ref: pp. 121-122) ───────────────────────
    # Three weighted axes per strategy. Highest composite wins.
    strategy_scores = {
        "full_autonomous_resolution": (
            decision_context["autonomy_level_score"] * 0.5
            + (1 - decision_context["escalation_threshold"]) * 0.3
            + (1 - decision_context["complexity_score"]) * 0.2
        ),
        "immediate_escalation": (
            decision_context["urgency_score"] * 0.4
            + (0.4 if not decision_context["agent_availability"] else 0.0)
            + decision_context["escalation_threshold"] * 0.2
        ),
        "guided_autonomous_resolution": 0.5,  # baseline fallback
    }

    # Round scores for readability
    strategy_scores = {k: round(v, 3) for k, v in strategy_scores.items()}

    # Strategy with highest score wins; ties favour guided resolution
    decision_context["strategy"] = max(
        strategy_scores, key=strategy_scores.get
    )
    decision_context["strategy_scores"] = strategy_scores

    log_success(
        f"Strategy selected: '{decision_context['strategy']}' "
        f"| Scores: {strategy_scores}"
    )
    return decision_context


# ── Quick test ─────────────────────────────────────────────────
test_decision = autonomous_reasoning(test_perception)
log_info(f"Decision intent: {test_decision['intent']}, "
         f"strategy: {test_decision['strategy']}")

In [ ]:
# ── Action Module: Task DAG & Execution ────────────────────────
# Ref: Section 5.1, Action: Autonomous execution and adaptation
#      (pp. 122-124)
#
# The action module translates reasoning into real-world outcomes.
# Tasks are organized as a dependency-aware DAG. Each node has an
# id, action, and depends_on list. Tasks with empty depends_on
# run immediately; downstream tasks wait until dependencies complete.

log_section("Action Module", chapter_ref="Section 5.1, pp. 122-124")


@fail_gracefully(
    fallback_value=[],
    chapter_ref="Section 5.1, pp. 123-124",
)
def create_autonomous_plan(decision_context: dict) -> list[dict]:
    """Create a dependency-aware task DAG based on intent.

    Builds a list of task dicts, each with 'id', 'action', and
    'depends_on' keys. Supports billing_issue, service_outage,
    and default resolution workflows.

    Args:
        decision_context: Decision dict from autonomous_reasoning().

    Returns:
        List of task dicts forming a DAG.

    Chapter Reference:
        Section 5.1, create_autonomous_plan() (pp. 123-124)
    """
    intent = decision_context.get("intent", "general_inquiry")

    # Ref: p. 123 — billing_issue DAG
    if intent == "billing_issue":
        plan = [
            {"id": "T1", "action": "fetch_account_details", "depends_on": []},
            {"id": "T2", "action": "analyze_billing_history", "depends_on": ["T1"]},
            {"id": "T3", "action": "identify_discrepancies", "depends_on": ["T2"]},
            {"id": "T4", "action": "calculate_adjustments", "depends_on": ["T3"]},
            {"id": "T5", "action": "generate_explanation", "depends_on": ["T3"]},
            {"id": "T6", "action": "apply_account_credits", "depends_on": ["T4"]},
            {"id": "T7", "action": "send_confirmation_email", "depends_on": ["T5", "T6"]},
            {"id": "T8", "action": "schedule_follow_up", "depends_on": ["T7"]},
        ]

    # Ref: p. 124 — service_outage DAG
    elif intent == "service_outage":
        plan = [
            {"id": "T1", "action": "check_service_status", "depends_on": []},
            {"id": "T2", "action": "identify_affected_areas", "depends_on": ["T1"]},
            {"id": "T3", "action": "estimate_resolution_time", "depends_on": ["T1"]},
            {"id": "T4", "action": "send_proactive_notifications", "depends_on": ["T2", "T3"]},
            {"id": "T5", "action": "monitor_restoration_progress", "depends_on": ["T1"]},
            {"id": "T6", "action": "update_customer_status", "depends_on": ["T4", "T5"]},
        ]

    # Ref: p. 124 — default resolution workflow
    else:
        plan = [
            {"id": "T1", "action": "analyze_issue", "depends_on": []},
            {"id": "T2", "action": "research_solutions", "depends_on": ["T1"]},
            {"id": "T3", "action": "implement_resolution", "depends_on": ["T2"]},
            {"id": "T4", "action": "verify_success", "depends_on": ["T3"]},
        ]

    log_info(f"Plan created for '{intent}': {len(plan)} tasks")
    for task in plan:
        deps = task["depends_on"] if task["depends_on"] else ["(none)"]
        log_info(f"  {task['id']}: {task['action']} → depends_on: {deps}")
    return plan


@fail_gracefully(
    fallback_value=[],
    chapter_ref="Section 5.1, pp. 122-123",
)
def execute_task(task: dict, completed: dict) -> dict:
    """Execute a single task from the DAG.

    Simulates task execution and returns a result dict.

    Args:
        task: Task dict with 'id', 'action', 'depends_on'.
        completed: Dict of already-completed task results.

    Returns:
        Result dict with 'task_id', 'action', 'status', 'timestamp'.

    Chapter Reference:
        Section 5.1, Action execution (pp. 122-123)
    """
    result = {
        "task_id": task["id"],
        "action": task["action"],
        "status": "completed",
        "timestamp": datetime.now().isoformat(),
    }
    log_success(f"Task {task['id']} ({task['action']}): completed")
    return result


@fail_gracefully(
    fallback_value=[],
    chapter_ref="Section 5.1, pp. 122-124",
)
def autonomous_action_execution(decision_context: dict) -> list[dict]:
    """Execute the autonomous action plan based on selected strategy.

    Dispatches to the appropriate execution path:
    - full_autonomous_resolution: build + execute full task DAG
    - immediate_escalation: prepare handoff package
    - guided_autonomous_resolution: execute with checkpoints

    Args:
        decision_context: Decision dict from autonomous_reasoning().

    Returns:
        List of task result dicts.

    Chapter Reference:
        Section 5.1, autonomous_action_execution() (pp. 122-123)
    """
    strategy = decision_context.get("strategy", "guided_autonomous_resolution")
    results = []

    if strategy == "full_autonomous_resolution":
        # Execute the full resolution workflow autonomously
        action_plan = create_autonomous_plan(decision_context)
        completed = {}

        # Topological execution: respect depends_on ordering
        remaining = list(action_plan)
        while remaining:
            ready = [
                t for t in remaining
                if all(d in completed for d in t["depends_on"])
            ]
            if not ready:
                log_error("Circular dependency detected in task DAG!")
                break
            for task in ready:
                result = execute_task(task, completed)
                completed[task["id"]] = result
                results.append(result)
                remaining.remove(task)

    elif strategy == "immediate_escalation":
        # Prepare comprehensive handoff to human agent
        log_warn("Escalation path: preparing handoff package")
        results.append({
            "task_id": "ESC-1",
            "action": "prepare_escalation_package",
            "status": "completed",
            "timestamp": datetime.now().isoformat(),
            "escalation_data": {
                "intent": decision_context.get("intent"),
                "priority": decision_context.get("priority"),
                "complexity": decision_context.get("complexity_score"),
            },
        })

    else:  # guided_autonomous_resolution
        # Execute with checkpoints for human oversight
        action_plan = create_autonomous_plan(decision_context)
        completed = {}
        remaining = list(action_plan)
        while remaining:
            ready = [
                t for t in remaining
                if all(d in completed for d in t["depends_on"])
            ]
            if not ready:
                break
            for task in ready:
                result = execute_task(task, completed)
                completed[task["id"]] = result
                results.append(result)
                remaining.remove(task)
        log_info("Guided resolution: all checkpoints passed")

    log_success(
        f"Action execution complete: {len(results)} tasks, "
        f"strategy='{strategy}'"
    )
    return results


# ── Quick test ─────────────────────────────────────────────────
test_results = autonomous_action_execution(test_decision)
log_info(f"Execution produced {len(test_results)} task results")

In [ ]:
# ── Safety Checks & Escalation Logic ──────────────────────────
# Ref: Section 5.1, Guardrails and safety mechanisms (p. 128)
#      and Escalation criteria (p. 131)
#
# Safety checks evaluate financial impact limits, data access
# permissions, and customer impact risk. Escalation uses a
# 5-factor weighted scoring system.

log_section("Safety & Escalation", chapter_ref="Section 5.1, pp. 128-131")


@fail_gracefully(
    fallback_value=(True, []),
    chapter_ref="Section 5.1, p. 128",
)
def autonomous_safety_check(
    decision_context: dict,
    planned_actions: list[dict],
) -> tuple[bool, list[str]]:
    """Evaluate planned actions against safety guardrails.

    Checks three categories:
    1. Financial impact limits (credit thresholds by tier)
    2. Data access permissions
    3. Customer impact risk assessment

    Args:
        decision_context: Decision dict from autonomous_reasoning().
        planned_actions: List of task dicts from create_autonomous_plan().

    Returns:
        Tuple of (is_safe: bool, violations: list[str]).

    Chapter Reference:
        Section 5.1, autonomous_safety_check() (p. 128)
    """
    safety_violations = []
    action_names = [a.get("action", "") for a in planned_actions]

    # 1. Financial impact limits (Ref: p. 128)
    if "apply_account_credits" in action_names:
        credit_limit = {"premium": 500, "enterprise": 2000, "standard": 100}
        tier = decision_context.get("context", {}).get("user_tier", "standard")
        max_credit = credit_limit.get(tier, 100)
        simulated_credit = 75  # Simulated credit amount
        if simulated_credit > max_credit:
            safety_violations.append("credit_limit_exceeded")
            log_error(f"Credit ${simulated_credit} exceeds {tier} limit ${max_credit}")
        else:
            log_success(f"Credit ${simulated_credit} within {tier} limit ${max_credit}")

    # 2. Data access permissions (Ref: p. 128)
    sensitive_actions = {"access_medical_records", "view_financial_data"}
    if sensitive_actions & set(action_names):
        safety_violations.append("insufficient_data_permissions")
        log_error("Sensitive data access attempted without explicit permission")
    else:
        log_success("Data access permissions: OK")

    # 3. Impact assessment (Ref: p. 128)
    impact_risk = decision_context.get("complexity_score", 0.3)
    threshold = decision_context.get("escalation_threshold", 0.5)
    if impact_risk > threshold:
        safety_violations.append("high_impact_risk")
        log_warn(f"Impact risk ({impact_risk}) exceeds threshold ({threshold})")
    else:
        log_success(f"Impact risk ({impact_risk}) within threshold ({threshold})")

    is_safe = len(safety_violations) == 0
    status = "PASSED" if is_safe else f"FAILED ({len(safety_violations)} violations)"
    log_info(f"Safety check: {status}")
    return is_safe, safety_violations


@fail_gracefully(
    fallback_value=None,
    chapter_ref="Section 5.1, p. 131",
)
def intelligent_escalation_decision(
    decision_context: dict,
    safety_results: tuple[bool, list[str]],
) -> dict | None:
    """Determine whether to escalate based on 5 weighted factors.

    Factors: safety violations, confidence threshold, complexity
    threshold, user preference, and business impact.

    Args:
        decision_context: Decision dict from autonomous_reasoning().
        safety_results: Tuple from autonomous_safety_check().

    Returns:
        Escalation package dict if escalation needed, else None.

    Chapter Reference:
        Section 5.1, intelligent_escalation_decision() (p. 131)
    """
    escalation_factors = {
        "safety_violations": not safety_results[0],
        "confidence_threshold": decision_context.get(
            "urgency_score", 0
        ) < 0.6,
        "complexity_threshold": decision_context.get(
            "complexity_score", 0
        ) > 0.7,
        "user_preference": False,  # Simulated: user has no preference
        "business_impact": decision_context.get("urgency_score", 0) > 0.8,
    }

    # Weighted escalation score
    weights = {
        "safety_violations": 0.30,
        "confidence_threshold": 0.20,
        "complexity_threshold": 0.25,
        "user_preference": 0.10,
        "business_impact": 0.15,
    }
    escalation_score = sum(
        weights[k] * (1.0 if v else 0.0)
        for k, v in escalation_factors.items()
    )
    escalation_score = round(escalation_score, 3)

    log_info(f"Escalation factors: {escalation_factors}")
    log_info(f"Escalation score: {escalation_score} "
             f"(threshold: {decision_context.get('escalation_threshold', 0.5)})")

    if escalation_score > decision_context.get("escalation_threshold", 0.5):
        log_warn("ESCALATION TRIGGERED — preparing handoff package")
        return {
            "escalation_score": escalation_score,
            "factors": escalation_factors,
            "decision_context_summary": {
                "intent": decision_context.get("intent"),
                "priority": decision_context.get("priority"),
                "strategy": decision_context.get("strategy"),
            },
        }

    log_success("No escalation needed — autonomous resolution continues")
    return None


# ── Quick test ─────────────────────────────────────────────────
test_plan = create_autonomous_plan(test_decision)
safety_ok, violations = autonomous_safety_check(test_decision, test_plan)
escalation = intelligent_escalation_decision(test_decision, (safety_ok, violations))
log_info(f"Safety: {safety_ok}, Escalation: {'triggered' if escalation else 'not needed'}")

In [ ]:
# ── AutonomousCustomerServiceAgent ─────────────────────────────
# Ref: Section 5.1, The continuous cognitive loop (pp. 124-126)
#
# This class integrates all components into a single cognitive loop:
# perception → reasoning → safety check → action → learning.
# It maintains session_memory, decision_history, and
# performance_metrics across interactions.

log_section("AutonomousCustomerServiceAgent", chapter_ref="Section 5.1, pp. 124-126")


class AutonomousCustomerServiceAgent:
    """Complete autonomous agent with the cognitive loop.

    Integrates perception, cognition, action, and learning into a
    continuous cycle. Maintains session memory and decision history
    for adaptation across interactions.

    Chapter Reference:
        Section 5.1, AutonomousCustomerServiceAgent (pp. 124-126)
    """

    def __init__(self) -> None:
        self.session_memory: dict = {}
        self.decision_history: list[dict] = []
        self.performance_metrics: dict = {"total_runs": 0, "successes": 0}

    def get_current_system_state(self) -> dict:
        """Return simulated system state for perception.

        Chapter Reference:
            Section 5.1, Perception (p. 120)
        """
        return {
            "active_sessions": 42,
            "system_load_pct": 65,
            "known_outages": ["NET-2026-0042"],
        }

    @fail_gracefully(
        fallback_value={"status": "cognitive_loop_failed"},
        chapter_ref="Section 5.1, pp. 124-126",
    )
    def cognitive_loop(
        self, user_message: str, context: dict
    ) -> list[dict]:
        """The complete autonomous decision-making cycle.

        Executes all four stages in sequence:
        1. Enhanced Perception
        2. Autonomous Reasoning
        3. Safety Check + Strategic Action Execution
        4. Learning and Adaptation

        Args:
            user_message: Raw user input.
            context: Session context dict.

        Returns:
            List of action result dicts.

        Chapter Reference:
            Section 5.1, cognitive_loop() (pp. 124-9)
        """
        log_section("Cognitive Loop START", chapter_ref="Section 5.1, pp. 124-9")

        # 1. Enhanced Perception
        system_state = self.get_current_system_state()
        perception_data = enhanced_perception(
            user_message, context, system_state
        )

        # 2. Autonomous Reasoning
        decision_context = autonomous_reasoning(perception_data)

        # 3. Safety Check + Action Execution
        planned_actions = create_autonomous_plan(decision_context)
        safety_ok, violations = autonomous_safety_check(
            decision_context, planned_actions
        )
        escalation = intelligent_escalation_decision(
            decision_context, (safety_ok, violations)
        )

        if escalation:
            decision_context["strategy"] = "immediate_escalation"

        results = autonomous_action_execution(decision_context)

        # 4. Learning and Adaptation
        self.learn_from_interaction(
            perception_data, decision_context, results
        )
        self.update_session_memory(
            context.get("user_id"), decision_context, results
        )

        log_section("Cognitive Loop END", chapter_ref="Section 5.1")
        return results

    def learn_from_interaction(
        self,
        perception_data: dict,
        decision_context: dict,
        results: list[dict],
    ) -> None:
        """Enhanced learning from interaction outcomes.

        Calculates success score, flags low-performing strategies,
        and stores the decision pattern for future reference.

        Chapter Reference:
            Section 5.1, learn_from_interaction() (pp. 125-126)
        """
        # Calculate success (Ref: p. 125)
        completed = sum(1 for r in results if r.get("status") == "completed")
        total = max(len(results), 1)
        success_score = round(completed / total, 2)

        self.performance_metrics["total_runs"] += 1
        if success_score >= 0.7:
            self.performance_metrics["successes"] += 1
            log_success(f"Learning: success_score={success_score} (above threshold)")
        else:
            log_warn(
                f"Learning: success_score={success_score} — "
                f"flagging strategy '{decision_context.get('strategy')}' "
                f"for improvement"
            )

        # Store decision pattern (Ref: p. 126)
        self.decision_history.append({
            "perception_summary": {
                "intent": decision_context.get("intent"),
                "sentiment": perception_data.get("sentiment"),
            },
            "decision": {
                "strategy": decision_context.get("strategy"),
                "scores": decision_context.get("strategy_scores"),
            },
            "outcome": success_score,
            "timestamp": datetime.now(),
        })
        log_info(f"Decision history: {len(self.decision_history)} entries")

    def update_session_memory(
        self,
        user_id: str | None,
        decision_context: dict,
        results: list[dict],
    ) -> None:
        """Update session memory for cross-interaction continuity.

        Chapter Reference:
            Section 5.1, update_session_memory() (p. 125)
        """
        if user_id:
            self.session_memory[user_id] = {
                "last_intent": decision_context.get("intent"),
                "last_strategy": decision_context.get("strategy"),
                "tasks_completed": len(results),
                "timestamp": datetime.now().isoformat(),
            }
            log_info(f"Session memory updated for '{user_id}'")


log_success("AutonomousCustomerServiceAgent class defined")

In [ ]:
# ── Case Study: Premium Business Customer Outage ───────────────
# Ref: Section 5.1, Case Study (p. 132)
#
# Scenario: A premium customer reports:
# "My business internet has been intermittent for two days,
#  and I have a critical presentation tomorrow."
#
# The agent must: perceive the urgency + business context,
# reason about strategy, validate safety, execute autonomously,
# and learn from the outcome.

log_section(
    "CASE STUDY: Business-Critical Outage",
    chapter_ref="Section 5.1, Case Study, p. 132",
)

agent = AutonomousCustomerServiceAgent()

results = agent.cognitive_loop(
    user_message=(
        "My business internet has been intermittent for two days, "
        "and I have a critical presentation tomorrow."
    ),
    context={"user_id": "premium_biz_123", "session": "new"},
)

# ── Summary ────────────────────────────────────────────────────
log_section("Case Study Summary")
log_info(f"Total tasks executed: {len(results)}")
log_info(f"Agent performance: {agent.performance_metrics}")
log_info(f"Session memory keys: {list(agent.session_memory.keys())}")
log_info(f"Decision history entries: {len(agent.decision_history)}")

if all(r.get("status") == "completed" for r in results):
    log_success("All tasks completed successfully — autonomous resolution achieved")
else:
    log_warn("Some tasks did not complete — review logs above")

### Analysis: Autonomous Decision-Making Agent

The case study above demonstrates the complete cognitive loop (Figure 5.1) in action:

- **Perception** (p. 124) — Enhanced context including system alerts, user tier, and agent availability.
- **Cognition** (p. 126) — LLM-powered reasoning with strategy selection and escalation assessment.
- **Action** — Task DAG execution with dependency ordering and safety checks.
- **Learning** — Success scoring feeds back to refine future autonomy thresholds.

> **📝 Note — Five-Factor Escalation Scoring (pp. 128–131)**  
> The agent uses five escalation factors: **safety_violations**, **confidence_threshold** (< 0.8), **complexity_threshold** (> 0.7), **user_preference**, and **business_impact** (> 0.6). A weighted escalation score is compared against the decision context's escalation threshold. This ensures that the agent can gracefully defer to a human operator for ambiguous situations, ethical dilemmas, or tasks beyond its capabilities.

> **📝 Note — Production Impact (p. 131)**  
> By combining structured prompts and conditional logic, such a bot can operate autonomously for up to **80% of user queries**, freeing human agents to focus on more nuanced and complex problems.


---
## 3. Planning Agent

**Chapter Ref:** Section 5.2 (pp. 131–137)

While Autonomous Decision-Making agents excel at real-time, single-step responses, many real-world challenges require breaking down complex objectives into sequences of smaller, manageable tasks executed over extended periods. The transition from reactive decision-making to proactive planning represents a significant leap in agent sophistication.

Planning agents master several interconnected capabilities: **decomposing** high-level goals into actionable subtasks, **sequencing** tasks appropriately, **allocating** resources, **monitoring** progress, and **adapting** plans dynamically.

> **📝 Note — Two Planning Paradigms (p. 132)**  
> Modern Planning agents leverage two complementary paradigms: **Symbolic planning foundations** (STRIPS, PDDL) which provide rigorous, mathematically grounded approaches for well-defined domains, and **LLM-powered dynamic planning** (Tree-of-Thought, Self-Ask) which introduces unprecedented flexibility for novel situations. The latter enables agents to decompose tasks without rigid pre-programming.

### Figure 5.2 — LLM-Powered Dynamic Planning


In [ ]:
# ── Figure 5.2 — Planning Agent Architecture (SVG) ──
# Ref: Section 5.2, p. 133

from IPython.display import SVG, display

fig_5_2 = """
<svg viewBox="0 0 700 380" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif">
  <defs><marker id="a52" markerWidth="10" markerHeight="7" refX="10" refY="3.5" orient="auto"><polygon points="0 0,10 3.5,0 7" fill="#555"/></marker></defs>
  <rect width="700" height="380" rx="12" fill="#f8f9fa" stroke="#dee2e6"/>
  <!-- Planning Agent Core -->
  <rect x="160" y="15" width="280" height="50" rx="8" fill="#e3f2fd" stroke="#42a5f5" stroke-width="2"/>
  <text x="300" y="37" text-anchor="middle" font-size="13" font-weight="bold" fill="#1565c0">Planning Agent Core</text>
  <text x="300" y="55" text-anchor="middle" font-size="9" fill="#555">Strategic Planning &amp; Orchestration Engine</text>
  <!-- External Environment -->
  <rect x="490" y="15" width="190" height="50" rx="8" fill="#fce4ec" stroke="#ef5350" stroke-width="1.5" stroke-dasharray="4,2"/>
  <text x="585" y="35" text-anchor="middle" font-size="10" font-weight="bold" fill="#c62828">External Environment</text>
  <text x="585" y="50" text-anchor="middle" font-size="8" fill="#555">Market shifts, Constraints</text>
  <!-- Row 2: Goal + Feedback + Phase 2 -->
  <rect x="30" y="90" width="150" height="40" rx="6" fill="#fff3e0" stroke="#ffa726" stroke-width="1.5"/>
  <text x="105" y="108" text-anchor="middle" font-size="11" font-weight="bold" fill="#e65100">High-Level Goal</text>
  <text x="105" y="122" text-anchor="middle" font-size="9" fill="#555">Launch Product</text>
  <rect x="210" y="90" width="140" height="40" rx="6" fill="#e0f2f1" stroke="#26a69a" stroke-width="1.5"/>
  <text x="280" y="115" text-anchor="middle" font-size="10" font-weight="bold" fill="#00695c">Execution Feedback</text>
  <rect x="380" y="90" width="140" height="40" rx="6" fill="#fff3e0" stroke="#ffa726" stroke-width="1.5"/>
  <text x="450" y="108" text-anchor="middle" font-size="10" font-weight="bold" fill="#e65100">Phase 2 Tasks</text>
  <text x="450" y="122" text-anchor="middle" font-size="9" fill="#555">Ad Setup</text>
  <!-- Plan Revision -->
  <text x="18" y="160" font-size="9" font-weight="bold" fill="#c62828">Plan Revision</text>
  <!-- Row 3: Phase Tasks -->
  <rect x="30" y="155" width="110" height="35" rx="5" fill="#e8f5e9" stroke="#66bb6a" stroke-width="1.5"/>
  <text x="85" y="177" text-anchor="middle" font-size="9" font-weight="bold" fill="#2e7d32">Phase Tasks</text>
  <rect x="155" y="155" width="110" height="35" rx="5" fill="#e8f5e9" stroke="#66bb6a" stroke-width="1.5"/>
  <text x="210" y="172" text-anchor="middle" font-size="9" font-weight="bold" fill="#2e7d32">Content</text>
  <text x="210" y="184" text-anchor="middle" font-size="9" font-weight="bold" fill="#2e7d32">Creation</text>
  <rect x="280" y="155" width="110" height="35" rx="5" fill="#e8f5e9" stroke="#66bb6a" stroke-width="1.5"/>
  <text x="335" y="172" text-anchor="middle" font-size="9" font-weight="bold" fill="#2e7d32">Design</text>
  <text x="335" y="184" text-anchor="middle" font-size="9" font-weight="bold" fill="#2e7d32">Banners</text>
  <rect x="405" y="155" width="110" height="35" rx="5" fill="#e8f5e9" stroke="#66bb6a" stroke-width="1.5"/>
  <text x="460" y="172" text-anchor="middle" font-size="9" font-weight="bold" fill="#2e7d32">Launch</text>
  <text x="460" y="184" text-anchor="middle" font-size="9" font-weight="bold" fill="#2e7d32">Event</text>
  <!-- Knowledge Base -->
  <rect x="545" y="140" width="140" height="50" rx="8" fill="#f3e5f5" stroke="#ab47bc" stroke-width="2"/>
  <text x="615" y="162" text-anchor="middle" font-size="10" font-weight="bold" fill="#6a1b9a">Agent Memory /</text>
  <text x="615" y="177" text-anchor="middle" font-size="10" font-weight="bold" fill="#6a1b9a">Knowledge Base</text>
  <!-- Row 4: Subtasks -->
  <rect x="30" y="210" width="110" height="30" rx="5" fill="#f1f8e9" stroke="#aed581" stroke-width="1"/>
  <text x="85" y="229" text-anchor="middle" font-size="8" fill="#555">Define Audience</text>
  <rect x="155" y="210" width="110" height="30" rx="5" fill="#f1f8e9" stroke="#aed581" stroke-width="1"/>
  <text x="210" y="229" text-anchor="middle" font-size="8" fill="#555">Write Blog Posts</text>
  <!-- Row 5: Tool/API -->
  <rect x="30" y="260" width="110" height="35" rx="5" fill="#fff9c4" stroke="#fdd835" stroke-width="1.5"/>
  <text x="85" y="276" text-anchor="middle" font-size="8" font-weight="bold" fill="#f57f17">Tool/API:</text>
  <text x="85" y="289" text-anchor="middle" font-size="8" fill="#555">Market Research</text>
  <rect x="155" y="260" width="110" height="35" rx="5" fill="#fff9c4" stroke="#fdd835" stroke-width="1.5"/>
  <text x="210" y="276" text-anchor="middle" font-size="8" font-weight="bold" fill="#f57f17">Tool/API:</text>
  <text x="210" y="289" text-anchor="middle" font-size="8" fill="#555">Content Gen API</text>
  <rect x="280" y="260" width="110" height="35" rx="5" fill="#fff9c4" stroke="#fdd835" stroke-width="1.5"/>
  <text x="335" y="276" text-anchor="middle" font-size="8" font-weight="bold" fill="#f57f17">Tool/API:</text>
  <text x="335" y="289" text-anchor="middle" font-size="8" fill="#555">Content Gen</text>
  <rect x="405" y="260" width="110" height="35" rx="5" fill="#fff9c4" stroke="#fdd835" stroke-width="1.5"/>
  <text x="460" y="276" text-anchor="middle" font-size="8" font-weight="bold" fill="#f57f17">Tool/API:</text>
  <text x="460" y="289" text-anchor="middle" font-size="8" fill="#555">Calendar/Booking</text>
  <!-- Arrows -->
  <line x1="300" y1="65" x2="300" y2="87" stroke="#555" stroke-width="1.5" marker-end="url(#a52)"/>
  <line x1="85" y1="130" x2="85" y2="152" stroke="#555" stroke-width="1" marker-end="url(#a52)"/>
  <line x1="85" y1="190" x2="85" y2="207" stroke="#555" stroke-width="1" marker-end="url(#a52)"/>
  <line x1="85" y1="240" x2="85" y2="257" stroke="#555" stroke-width="1" marker-end="url(#a52)"/>
  <!-- Caption -->
  <text x="350" y="320" text-anchor="middle" font-size="10" fill="#777" font-style="italic">Figure 5.2 — LLM-powered dynamic planning: from high-level goal to tool invocations (p. 133)</text>
  <text x="350" y="340" text-anchor="middle" font-size="9" fill="#999">Environmental awareness enables proactive adaptation; execution feedback triggers plan revision</text>
</svg>
"""
display(SVG(fig_5_2))


In [ ]:
# ── PlanningAgent Class ────────────────────────────────────────
# Ref: Section 5.2, Planning Agent (pp. 131-137)
#
# Implements hierarchical task decomposition with dependency
# resolution via topological sort. The agent decomposes a
# high-level goal into phased subtasks, executes them in
# dependency order, monitors progress, and can revise the plan
# based on feedback.

log_section("Planning Agent", chapter_ref="Section 5.2, pp. 131-137")


class PlanningAgent:
    """Agent for orchestrating complex, multi-step workflows.

    Decomposes high-level goals into hierarchical task plans,
    executes them respecting dependency order via topological
    sort, and supports adaptive plan revision.

    Chapter Reference:
        Section 5.2, The Planning Agent (pp. 131-137)
    """

    def __init__(self, llm_client_ref) -> None:
        """Initialize with an LLM client reference.

        Args:
            llm_client_ref: MockLLM or real LLM client instance.
        """
        self.llm = llm_client_ref
        self.execution_log: list[dict] = []
        self.plan_revisions: int = 0

    @fail_gracefully(
        fallback_value=[],
        chapter_ref="Section 5.2, pp. 133-20",
    )
    def decompose_goal(self, goal: str) -> list[dict]:
        """Decompose a high-level goal into a hierarchical task DAG.

        Uses the LLM to identify the goal type, then builds a
        structured task plan matching Figure 5.2's architecture.

        Args:
            goal: High-level objective string.

        Returns:
            List of task dicts with 'id', 'action', 'phase',
            'depends_on', and 'tools' keys.

        Chapter Reference:
            Section 5.2, Strategic task decomposition (pp. 133-20)
        """
        response = self.llm.generate(goal)
        log_info(f"Goal decomposition: '{goal}'")
        log_info(f"LLM identified strategy: '{response.strategy}'")

        # Ref: Figure 5.2 — marketing campaign task hierarchy
        if response.intent == "planning_request":
            plan = [
                # Phase 1: Research (Ref: Figure 5.2, left branch)
                {
                    "id": "P1-T1", "phase": 1,
                    "action": "define_target_audience",
                    "depends_on": [],
                    "tools": ["market_research_platform"],
                },
                {
                    "id": "P1-T2", "phase": 1,
                    "action": "competitive_analysis",
                    "depends_on": [],
                    "tools": ["market_research_platform"],
                },
                # Phase 2: Content Creation (Ref: Figure 5.2, center)
                {
                    "id": "P2-T1", "phase": 2,
                    "action": "write_blog_posts",
                    "depends_on": ["P1-T1"],
                    "tools": ["content_generation_api"],
                },
                {
                    "id": "P2-T2", "phase": 2,
                    "action": "design_banners",
                    "depends_on": ["P1-T1"],
                    "tools": ["content_generation_api"],
                },
                # Phase 3: Distribution (Ref: Figure 5.2, right)
                {
                    "id": "P3-T1", "phase": 3,
                    "action": "setup_ad_campaigns",
                    "depends_on": ["P2-T1", "P2-T2"],
                    "tools": ["ad_platform_api"],
                },
                {
                    "id": "P3-T2", "phase": 3,
                    "action": "schedule_launch_event",
                    "depends_on": ["P1-T2"],
                    "tools": ["calendar_booking_api"],
                },
                # Phase 4: Monitoring (Ref: Figure 5.2, feedback loop)
                {
                    "id": "P4-T1", "phase": 4,
                    "action": "monitor_campaign_metrics",
                    "depends_on": ["P3-T1", "P3-T2"],
                    "tools": ["analytics_dashboard"],
                },
                {
                    "id": "P4-T2", "phase": 4,
                    "action": "adaptive_revision_check",
                    "depends_on": ["P4-T1"],
                    "tools": [],
                },
            ]
        else:
            # Generic planning decomposition
            plan = [
                {"id": "G-T1", "phase": 1, "action": "analyze_requirements",
                 "depends_on": [], "tools": []},
                {"id": "G-T2", "phase": 2, "action": "design_solution",
                 "depends_on": ["G-T1"], "tools": []},
                {"id": "G-T3", "phase": 3, "action": "implement_solution",
                 "depends_on": ["G-T2"], "tools": []},
                {"id": "G-T4", "phase": 4, "action": "validate_results",
                 "depends_on": ["G-T3"], "tools": []},
            ]

        log_success(f"Decomposed into {len(plan)} tasks across "
                    f"{max(t['phase'] for t in plan)} phases")
        return plan

    @fail_gracefully(
        fallback_value=[],
        chapter_ref="Section 5.2, pp. 134-135",
    )
    def execute_plan(self, plan: list[dict]) -> list[dict]:
        """Execute a task plan respecting dependency order.

        Uses topological sort to determine execution order.
        Tasks with satisfied dependencies execute in phase order.
        Each task is individually wrapped for fault tolerance.

        Args:
            plan: Task DAG from decompose_goal().

        Returns:
            List of execution result dicts.

        Chapter Reference:
            Section 5.2, Dynamic execution and monitoring (pp. 134-135)
        """
        results = []
        completed: dict[str, dict] = {}
        remaining = list(plan)

        log_info("Beginning plan execution (topological order)...")

        while remaining:
            # Find tasks whose dependencies are all satisfied
            ready = [
                t for t in remaining
                if all(d in completed for d in t["depends_on"])
            ]
            if not ready:
                log_error("Deadlock: unresolvable dependencies remain")
                break

            for task in ready:
                result = self._execute_single_task(task)
                completed[task["id"]] = result
                results.append(result)
                remaining.remove(task)

                # Monitor progress (Ref: p. 134)
                self.monitor_progress(plan, completed)

        log_success(f"Plan execution complete: {len(results)}/{len(plan)} tasks")
        return results

    @fail_gracefully(
        fallback_value={"status": "task_failed"},
        chapter_ref="Section 5.2, p. 136",
    )
    def _execute_single_task(self, task: dict) -> dict:
        """Execute a single task and return its result.

        Args:
            task: Task dict from the plan.

        Returns:
            Result dict with status, timing, and metadata.

        Chapter Reference:
            Section 5.2, Dynamic execution (p. 136)
        """
        tools_str = ", ".join(task["tools"]) if task["tools"] else "none"
        log_info(f"  Phase {task['phase']} | {task['id']}: "
                 f"{task['action']} (tools: {tools_str})")

        result = {
            "task_id": task["id"],
            "phase": task["phase"],
            "action": task["action"],
            "status": "completed",
            "timestamp": datetime.now().isoformat(),
        }
        log_success(f"  {task['id']}: completed")
        self.execution_log.append(result)
        return result

    def monitor_progress(
        self, plan: list[dict], completed: dict
    ) -> None:
        """Track execution progress and log phase completion.

        Chapter Reference:
            Section 5.2, Dynamic execution and monitoring (p. 134)
        """
        total = len(plan)
        done = len(completed)
        pct = round(done / total * 100)

        # Check if a phase just completed
        completed_phases = set()
        for t in plan:
            if t["id"] in completed:
                completed_phases.add(t["phase"])

        # Log phase completions
        for phase in sorted(completed_phases):
            phase_tasks = [t for t in plan if t["phase"] == phase]
            if all(t["id"] in completed for t in phase_tasks):
                phase_actions = [t["action"] for t in phase_tasks]
                if done <= len(phase_tasks) or done == total:
                    log_info(f"  Progress: {pct}% ({done}/{total}) "
                             f"— Phase {phase} complete")

    @fail_gracefully(
        fallback_value=None,
        chapter_ref="Section 5.2, p. 134",
    )
    def revise_plan(
        self, plan: list[dict], feedback: str
    ) -> list[dict]:
        """Revise the plan based on execution feedback.

        Demonstrates the adaptive intelligence capability
        where the agent modifies strategies based on new
        information or changing conditions.

        Args:
            plan: Current task plan.
            feedback: Feedback string describing what changed.

        Returns:
            Revised plan (or original if no changes needed).

        Chapter Reference:
            Section 5.2, Adaptive intelligence and learning (p. 134)
        """
        self.plan_revisions += 1
        log_warn(f"Plan revision #{self.plan_revisions}: {feedback}")

        # Simulate adding a corrective task
        max_phase = max(t["phase"] for t in plan)
        revision_task = {
            "id": f"REV-{self.plan_revisions}",
            "phase": max_phase + 1,
            "action": f"corrective_action_{self.plan_revisions}",
            "depends_on": [plan[-1]["id"]],
            "tools": [],
        }
        revised = plan + [revision_task]
        log_info(f"Added corrective task: {revision_task['id']}")
        return revised


log_success("PlanningAgent class defined")

In [ ]:
# ── Demo: Marketing Campaign Planning ──────────────────────────
# Ref: Section 5.2, Figure 5.2 (p. 133)
#
# Scenario: "Launch Product Marketing Campaign"
# The agent decomposes this into phased tasks, executes them
# in dependency order, demonstrates the feedback loop, and
# shows adaptive plan revision.

log_section(
    "DEMO: Marketing Campaign Planning",
    chapter_ref="Section 5.2, Figure 5.2, p. 133",
)

planner = PlanningAgent(llm_client)

# Step 1: Decompose the goal
campaign_plan = planner.decompose_goal("Launch Product Marketing Campaign")

print()
log_info("Task DAG overview:")
for task in campaign_plan:
    deps = task["depends_on"] if task["depends_on"] else ["(start)"]
    log_info(f"  Phase {task['phase']} | {task['id']}: "
             f"{task['action']} → depends_on: {deps}")

# Step 2: Execute the plan
print()
log_section("Executing Plan", chapter_ref="Section 5.2, pp. 134-135")
campaign_results = planner.execute_plan(campaign_plan)

# Step 3: Simulate adaptive revision (Ref: p. 134)
print()
log_section("Adaptive Revision", chapter_ref="Section 5.2, p. 134")
revised_plan = planner.revise_plan(
    campaign_plan,
    feedback="Competitor launched similar campaign — need differentiation"
)

# Step 4: Execute the revision
revision_results = planner.execute_plan(
    [t for t in revised_plan if t["id"].startswith("REV-")]
)

# Summary
print()
log_section("Planning Demo Summary")
log_info(f"Original tasks: {len(campaign_plan)}")
log_info(f"After revision: {len(revised_plan)} (+{len(revised_plan) - len(campaign_plan)})")
log_info(f"Total executions: {len(planner.execution_log)}")
log_success("Marketing campaign planning demo complete")

### Analysis: Planning Agent

The demo above illustrates the **LLM-powered dynamic planning** paradigm
from Figure 5.2. Key observations:

1. **Hierarchical decomposition** broke "Launch Product Marketing Campaign"
   into 4 phases with 8 tasks, matching the figure's structure:
   research → content creation → distribution → monitoring.

2. **Topological execution** ensured that Phase 2 tasks (blog posts,
   banners) waited for Phase 1 (audience definition) to complete.
   Parallel tasks within the same phase executed together.

3. **Adaptive revision** demonstrated the feedback loop — when a
   competitor threat was detected, the agent appended a corrective task
   to the plan without disrupting completed work.

#### Planning vs. Decision-Making (Table 5.1)

| Capability | Autonomous Decision-Maker | Planning Agent |
|---|---|---|
| **Focus** | Immediate, tactical responses | Strategic, long-term goal achievement |
| **Scope** | Single decisions at a time | Multi-step, coordinated processes |
| **Adaptation** | Heuristics and conditional logic | Dynamic revision and learning loops |
| **Integration** | Direct API calls for specific actions | Orchestrated tool workflows |
| **Memory Use** | Contextual within current session | Persistent, cross-project learning |

Ref: Section 5.2 (pp. 131-137), Table 5.1 (pp. 135-136)

---
## 4. Memory-Augmented Agent

**Chapter Ref:** Section 5.3 (pp. 135–144)

Autonomous agents can make decisions and formulate plans, but a critical limitation remains: how can they retain context and learn from past experiences beyond the current interaction? Many foundational agents operate solely in the present. **Memory-Augmented agents** incorporate mechanisms that preserve context over time, significantly improving personalization, coherence, and long-term reasoning.

### Three Types of Memory

| Memory Type | Role | Trigger Condition | Failure Mode if Skipped |
|---|---|---|---|
| **Working** | Short-term session buffer (LLM prompt) | Active session, current turn | Loss of immediate conversational coherence |
| **Episodic** | Historical interactions & events | Cross-session continuity needed | Agent treats each session as new, breaking personalization |
| **Semantic** | Structured facts & world knowledge | Domain knowledge required | Hallucinated or stale answers |

> **📝 Note — Memory Selection Guide (p. 136)**  
> Use this guide to select which memory system to query: **Working memory** when the LLM context window is being populated directly during an active session. **Episodic memory** via vector similarity search over interaction history when cross-session continuity is needed. **Semantic memory** via keyword or semantic search over a knowledge base when factual or domain knowledge is required that isn't answerable from the session alone.

> **📝 Note — Vector Database Options (pp. 137–138)**  
> Practical implementations include: **Chroma** (well-suited for local development and prototyping), **Pinecone** (managed cloud service optimized for production-scale retrieval), and **Weaviate** (open source with hybrid keyword and vector search support).


In [ ]:
# ── MemoryAugmentedAgent Class ─────────────────────────────────
# Ref: Section 5.3, Memory-Augmented Agent (pp. 138-139)
#
# Core logic: retrieve relevant episodic memories to provide
# context for the current query, generate a response with the
# LLM, then store the new interaction for future use.

log_section("Memory-Augmented Agent", chapter_ref="Section 5.3, pp. 135-144")


class MemoryAugmentedAgent:
    """Agent with episodic memory for context-aware interactions.

    Retrieves relevant past interactions from a vector database,
    injects them into the LLM prompt as context, generates a
    response, and stores the new interaction for future retrieval.

    Chapter Reference:
        Section 5.3, MemoryAugmentedAgent (pp. 138-139)
    """

    def __init__(self, llm_client_ref, vector_db_ref) -> None:
        """Initialize with LLM client and vector database.

        Args:
            llm_client_ref: MockLLM or real LLM client.
            vector_db_ref: MockVectorDB or real vector DB.
        """
        self.llm = llm_client_ref
        self.episodic_memory = vector_db_ref
        self.interaction_count: int = 0

    @fail_gracefully(
        fallback_value="I apologize, but I was unable to process your request.",
        chapter_ref="Section 5.3, pp. 138-139",
    )
    def process_interaction(
        self, user_id: str, user_query: str
    ) -> str:
        """Process a user query with episodic memory augmentation.

        Steps:
        1. Retrieve relevant past interactions from episodic memory
        2. Construct a prompt that includes retrieved memories
        3. Generate a response via the LLM
        4. Store the current interaction for future use

        Args:
            user_id: Unique user identifier.
            user_query: Current user query text.

        Returns:
            Generated response string.

        Chapter Reference:
            Section 5.3, process_interaction() (pp. 138-139)
        """
        self.interaction_count += 1
        log_info(f"Interaction #{self.interaction_count} for user '{user_id}'")

        # 1. Retrieve relevant past interactions (Ref: p. 141-26)
        log_info("Step 1: Retrieving episodic memories...")
        relevant_memories = self.episodic_memory.search(
            query=user_query,
            user_id=user_id,
            limit=3,
        )

        if relevant_memories:
            log_success(f"Retrieved {len(relevant_memories)} relevant memories:")
            for i, mem in enumerate(relevant_memories):
                log_info(
                    f"  Memory {i+1}: '{mem['query'][:50]}...' "
                    f"(relevance: {mem.get('relevance_score', 'N/A')})"
                )
        else:
            log_warn("No relevant memories found — first interaction")

        # 2. Construct prompt with memory context (Ref: pp. 138-139)
        memory_context = self._format_memories(relevant_memories)
        prompt = (
            f"You are a personalized healthcare assistant.\n"
            f"Here is the user's current query: \"{user_query}\"\n"
            f"For context, here are relevant past interactions "
            f"with this user:\n{memory_context}\n"
            f"Provide a helpful and context-aware response."
        )
        log_info("Step 2: Prompt constructed with memory context")

        # 3. Generate response (Ref: p. 138)
        log_info("Step 3: Generating response...")
        response = self.llm.generate(user_query)
        response_text = response.content

        # 4. Store interaction in episodic memory (Ref: p. 138)
        log_info("Step 4: Storing interaction in episodic memory...")
        interaction_record = {
            "user_id": user_id,
            "query": user_query,
            "response_summary": response_text[:100],
            "timestamp": datetime.now().isoformat(),
        }
        self.episodic_memory.add(interaction_record)

        log_success(f"Interaction #{self.interaction_count} complete")
        return response_text

    def _format_memories(self, memories: list[dict]) -> str:
        """Format retrieved memories for prompt injection.

        Args:
            memories: List of memory dicts from vector DB search.

        Returns:
            Formatted string for inclusion in LLM prompt.

        Chapter Reference:
            Section 5.3, prompt construction (p. 139)
        """
        if not memories:
            return "(No prior interactions found)"

        lines = []
        for mem in memories:
            lines.append(
                f"- [{mem.get('timestamp', 'unknown')}] "
                f"Query: '{mem.get('query', '')}' → "
                f"Response: '{mem.get('response_summary', '')}'"
            )
        return "\n".join(lines)

    def get_prompt_preview(
        self, user_id: str, user_query: str
    ) -> str:
        """Show the full prompt that would be sent to the LLM.

        Useful for readers to understand how memory retrieval
        augments generation — without actually calling the LLM.

        Args:
            user_id: User identifier for memory lookup.
            user_query: Current query text.

        Returns:
            The complete prompt string.

        Chapter Reference:
            Section 5.3, Prompt inspection (p. 139)
        """
        memories = self.episodic_memory.search(
            query=user_query, user_id=user_id, limit=3
        )
        memory_context = self._format_memories(memories)
        return (
            f"You are a personalized healthcare assistant.\n"
            f"Here is the user's current query: \"{user_query}\"\n"
            f"For context, here are relevant past interactions "
            f"with this user:\n{memory_context}\n"
            f"Provide a helpful and context-aware response."
        )


log_success("MemoryAugmentedAgent class defined")

In [ ]:
# ── Healthcare Case Study: 3-Turn Conversation ─────────────────
# Ref: Section 5.3, Case Study: A personalized healthcare
#      assistant (pp. 139-140)
#
# Turn 1: Patient reports fatigue (new interaction)
# Turn 2: Patient follows up about diet change (retrieves Turn 1)
# Turn 3: Patient mentions iron deficiency (retrieves both priors)

log_section(
    "CASE STUDY: Personalized Healthcare Assistant",
    chapter_ref="Section 5.3, pp. 139-140",
)

# Reset vector DB to start fresh with pre-seeded records
vector_db.reset()
health_agent = MemoryAugmentedAgent(llm_client, vector_db)

# ── Turn 1: Initial fatigue report ─────────────────────────────
log_section("Turn 1: Initial Fatigue Report", chapter_ref="p. 139")
response_1 = health_agent.process_interaction(
    user_id="patient_42",
    user_query="I've been feeling very fatigued lately",
)
log_info(f"Agent response: {response_1[:100]}...")

# ── Turn 2: Follow-up about diet change ────────────────────────
print()
log_section("Turn 2: Diet Change Follow-Up", chapter_ref="p. 139")
response_2 = health_agent.process_interaction(
    user_id="patient_42",
    user_query="That diet change you suggested helped a bit, but I'm still tired",
)
log_info(f"Agent response: {response_2[:100]}...")

# ── Turn 3: Iron deficiency update ─────────────────────────────
print()
log_section("Turn 3: Iron Deficiency Update", chapter_ref="p. 142")
response_3 = health_agent.process_interaction(
    user_id="patient_42",
    user_query="My allergist said I'm low on iron, what should I do?",
)
log_info(f"Agent response: {response_3[:100]}...")

# Summary
print()
log_section("Healthcare Case Study Summary")
log_info(f"Total interactions: {health_agent.interaction_count}")
log_success("Memory accumulation demonstrated across 3 turns")

In [ ]:
# ── Prompt Inspection ──────────────────────────────────────────
# Ref: Section 5.3, Prompt construction (p. 139)
#
# Display the actual prompt that would be sent to the LLM for
# Turn 3, showing how retrieved episodic memories are injected
# into the context window.

log_section("Prompt Inspection", chapter_ref="Section 5.3, p. 139")

prompt_preview = health_agent.get_prompt_preview(
    user_id="patient_42",
    user_query="My allergist said I'm low on iron, what should I do?",
)

log_info("Full prompt sent to LLM (with injected memories):")
print()
print("=" * 60)
print(prompt_preview)
print("=" * 60)
print()
log_success(
    "This demonstrates how episodic memory retrieval augments "
    "the LLM's context, enabling personalized responses."
)

### Analysis: Memory-Augmented Agent

The healthcare case study demonstrates how memory transforms agent behavior:

**Turn 1** — The agent processes a new fatigue report. With pre-seeded
episodic memory, it retrieves relevant prior records and stores this
interaction for future reference.

**Turn 2** — When the patient follows up, `process_interaction()` searches
episodic memory and retrieves Turn 1's record because of semantic overlap
("fatigue" ↔ "tired", "diet"). The retrieved context is injected directly
into the LLM prompt, enabling a coherent response that acknowledges the
ongoing experience.

**Turn 3** — The agent now retrieves both prior turns, demonstrating
**memory accumulation**. The prompt inspection cell reveals the exact
prompt structure described in the book (p. 139): user query + retrieved
memories formatted as context.

#### Memory Consolidation for Scalability

For production systems, the chapter recommends summarizing long interaction
histories into concise patient profiles (p. 142). For example:
> `Patient A, Type 2 Diabetes, responded well to diet changes (2026-01-15),
> struggled with medication adherence (2026-03-20).`

This ensures the vector database remains efficient as interaction
history grows.

Ref: Section 5.3 (pp. 135-144), Figure 5.3

---
## 5. Comparative Analysis

**Chapter Ref:** Section 5.4 (pp. 141–142)

Each foundational cognitive architecture excels in its specific domain. However, their true potential emerges through **strategic combinations**. These architectures are not mutually exclusive — they are complementary building blocks.

### Table 5.2 — Comparative Analysis

| Agent Type | Strength | Ideal Use Case | Limitations |
|---|---|---|---|
| **Autonomous Decision-Making** | Real-time response and flexibility | Customer service, triage | Risk of hallucinations; lacks deep context |
| **Planning Agent** | Structured execution and adaptability | Project orchestration, complex workflows | Slower execution; requires control logic |
| **Memory-Augmented Agent** | Long-term coherence and personalization | CRM systems, task handoff, continuity | Complex maintenance; sensitive to retrieval design |

### Figure 5.4 — Integrated Architecture

The demo below combines all three agent types: the **Decision-Making agent** handles the immediate query, the **Planning agent** creates a multi-step resolution plan, and the **Memory-Augmented agent** retrieves relevant history to personalize the response.


In [ ]:
# ── Integrated Demo: All Three Agent Types ─────────────────────
# Ref: Section 5.4, Figure 5.4 (p. 142)
#
# Scenario: A premium customer has a complex issue that requires
# all three architectures working together:
# 1. Decision agent handles initial triage
# 2. Planning agent decomposes follow-up tasks
# 3. Memory agent maintains context across the interaction

log_section(
    "INTEGRATED DEMO: All Three Architectures",
    chapter_ref="Section 5.4, Figure 5.4, p. 142",
)

# ── Step 1: Decision Agent — Initial Triage ────────────────────
log_section("Step 1: Decision Agent — Triage")

decision_agent = AutonomousCustomerServiceAgent()
triage_results = decision_agent.cognitive_loop(
    user_message="My internet is down and I need to file a billing dispute",
    context={"user_id": "premium_biz_123", "session": "integrated_demo"},
)
triage_intent = decision_agent.decision_history[-1]["perception_summary"]["intent"]
log_info(f"Triage result: intent='{triage_intent}', "
         f"tasks={len(triage_results)}")

# ── Step 2: Planning Agent — Follow-Up Decomposition ───────────
print()
log_section("Step 2: Planning Agent — Decomposition")

integrated_planner = PlanningAgent(llm_client)
followup_plan = integrated_planner.decompose_goal(
    "Plan follow-up investigation for customer billing and connectivity issues"
)
followup_results = integrated_planner.execute_plan(followup_plan)
log_info(f"Planning result: {len(followup_results)} tasks completed")

# ── Step 3: Memory Agent — Context Preservation ────────────────
print()
log_section("Step 3: Memory Agent — Context Preservation")

# Reset and use fresh memory agent for this demo
vector_db.reset()
memory_agent = MemoryAugmentedAgent(llm_client, vector_db)

# Store the triage outcome as an episodic record
memory_agent.episodic_memory.add({
    "user_id": "premium_biz_123",
    "query": "Internet down + billing dispute (integrated triage)",
    "response_summary": (
        f"Triage: {triage_intent}, "
        f"{len(triage_results)} actions executed, "
        f"follow-up plan: {len(followup_results)} tasks"
    ),
    "timestamp": datetime.now().isoformat(),
})

# Later interaction — memory retrieves the triage context
memory_response = memory_agent.process_interaction(
    user_id="premium_biz_123",
    user_query="What was done about my internet and billing issue?",
)
log_info(f"Memory-augmented response: {memory_response[:80]}...")

# ── Integration Summary ────────────────────────────────────────
print()
log_section("Integration Summary", chapter_ref="Figure 5.4, p. 142")
log_success("Decision Agent → triaged the initial request")
log_success("Planning Agent → decomposed follow-up into structured tasks")
log_success("Memory Agent   → preserved context for future interactions")
log_info(
    "This demonstrates how the three architectures complement each other "
    "when combined (Figure 5.4)"
)

### When to Use Which Architecture

- **Use Decision-Making** when you need fast, reactive responses to
  individual queries — customer service chatbots, alerting systems, triage.

- **Use Planning** when the task requires multi-step coordination with
  dependencies — project orchestration, workflow automation, campaign execution.

- **Use Memory-Augmented** when continuity across sessions matters —
  personalized assistants, CRM systems, healthcare follow-ups.

- **Combine all three** for complex scenarios where an initial triage
  leads to a structured plan that must be remembered across sessions —
  exactly the pattern shown in Figure 5.4.

Ref: Section 5.4 (pp. 141-142), Table 5.2

---
## 6. Engineering Best Practices & Wrap-Up

**Chapter Ref:** Section 5.5 (pp. 142–144)

Building reliable intelligent agents requires adherence to sound engineering principles throughout the development lifecycle.

> **📝 Note — Scalability Principles (p. 143)**  
> **Start modular:** Separate capabilities into distinct modules (perception, memory, planning, action). This separation of concerns simplifies development and debugging.  
> **Log everything:** Implement comprehensive logging at every stage of the agent's decision loop. Tools like LangSmith are critical for understanding *why* an agent made a particular decision.

> **📝 Note — Context Management (p. 143)**  
> **Avoid prompt bloat:** LLMs have finite context windows. Feeding them excessive or irrelevant information degrades performance, increases latency, and incurs higher costs. Optimize retrieval pipelines to provide only the most relevant context — techniques like re-ranking (Chapter 6) are crucial.

> **📝 Note — Robustness (p. 143)**  
> Design systems with clear fallbacks and escalation paths for undecidable or unsafe conditions. This might mean defaulting to a human operator, requesting clarification, or rolling back an action. Every tool call in this repository is wrapped in the `@fail_gracefully` decorator for exponential-backoff retries.

> **What's Next:** Chapter 6 turns to *Information Retrieval and Knowledge Agents*, expanding these ideas further by exploring how agents access and leverage vast external knowledge to enrich their decision-making.


In [ ]:
# ── Final Status Report ────────────────────────────────────────
# Simulation summary: all agent runs, success/failure counts,
# mock vs live mode status.

log_section("FINAL STATUS REPORT", chapter_ref="Chapter 5 Summary")

# Gather metrics
total_agent_runs = 0
if 'agent' in dir():
    total_agent_runs += agent.performance_metrics.get("total_runs", 0)
if 'decision_agent' in dir():
    total_agent_runs += decision_agent.performance_metrics.get("total_runs", 0)

total_planning_tasks = 0
if 'planner' in dir():
    total_planning_tasks += len(planner.execution_log)
if 'integrated_planner' in dir():
    total_planning_tasks += len(integrated_planner.execution_log)

total_memory_interactions = 0
if 'health_agent' in dir():
    total_memory_interactions += health_agent.interaction_count
if 'memory_agent' in dir():
    total_memory_interactions += memory_agent.interaction_count

log_info(f"Mode: {'SIMULATION' if SIMULATION_MODE else 'LIVE'}")
log_info(f"Decision Agent runs: {total_agent_runs}")
log_info(f"Planning Agent tasks executed: {total_planning_tasks}")
log_info(f"Memory Agent interactions: {total_memory_interactions}")

# Verify all architectures demonstrated
architectures_ok = (
    total_agent_runs > 0
    and total_planning_tasks > 0
    and total_memory_interactions > 0
)

if architectures_ok:
    print()
    log_success("=" * 50)
    log_success("  ALL THREE ARCHITECTURES DEMONSTRATED SUCCESSFULLY")
    log_success("=" * 50)
    print()
    log_info(
        "This notebook implemented the three foundational cognitive "
        "architectures from Chapter 5: Autonomous Decision-Making, "
        "Planning, and Memory-Augmented agents."
    )
else:
    log_error("Some architectures were not demonstrated — check cells above")

### Further Reading

- **Chapter 1** — The cognitive loop foundation (perception → reasoning → action → learning)
- **Chapter 3** — *The Art of Agent Prompting* — prompt engineering for chain-of-thought planning
- **Chapter 4** — *Agent Deployment and Responsible Development* — production safety and security
- **Chapter 6** — *Information Retrieval and Knowledge Agents* — advanced retrieval for memory systems
- **Chapter 7** — *Multi-agent systems* — combining agents into collaborative workflows

**Book Repository:**
[PacktPublishing/30-Agents-Every-AI-Engineer-Must-Build](https://github.com/PacktPublishing/30-Agents-Every-AI-Engineer-Must-Build)

---

*Chapter 5: Foundational Cognitive Architectures*
*Author: Imran Ahmad | Publisher: Packt Publishing, 2026*